# Ecuador Desigual 🇪🇨 — Exploración del Censo 2022

**Fase I: Provincia de El Oro**  
**Tema:** conexión de viviendas particulares ocupadas a la red pública de alcantarillado.  
**Fuente:** INEC, Censo de Población y Vivienda 2022, tabulado `3.1.csv`.

**Pregunta 1:** ¿Cómo varía la cobertura de alcantarillado público entre los cantones de El Oro?  
**Pregunta 2 (H2):** ¿Es menor la cobertura rural que la urbana en los cantones de El Oro?

> La conexión a la red pública de alcantarillado no equivale, por sí sola, a saneamiento adecuado. Este análisis es descriptivo: no identifica causas.

## 1. Cargar e inspeccionar los datos

El CSV incluye metadatos y encabezados en varias filas; los registros comienzan en la posición 11. No eliminamos valores ausentes sin revisar primero su significado.

In [2]:
import pandas as pd

ruta_csv = "../data/raw/archivos/3.1.csv"  # Ajusta la ruta si moviste el archivo.
df = pd.read_csv(ruta_csv, encoding="cp1252", header=None)

display(df.iloc[8:16, 1:12])

,1,2,3,4,5,6,7,8,9,10,11
8,"Provincia, cantón y área",NaN,NaN,Número total de viviendas particulares ocupadas,Tipo de servicio higiénico en la vivienda,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,"Inodoro o escusado, conectado a red pública de...","Inodoro o escusado, conectado a pozo séptico","Inodoro o escusado, conectado a biodigestor","Inodoro o escusado, conectado a pozo ciego","Inodoro o escusado, con descarga directa al ma...",Letrina,No tiene
10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Total Nacional,Nacional,Nacional,"5,062,650","3,330,851","1,222,743","10,019","213,824","48,483","59,285","177,445"
12,Total Nacional,Nacional,Urbana,"3,238,614","2,677,671","456,526","4,072","43,999","19,149","6,526","30,671"
13,Total Nacional,Nacional,Rural,"1,824,036","653,180","766,217","5,947","169,825","29,334","52,759","146,774"
14,Azuay,Total Azuay,Total Azuay,"242,168","174,655","52,616","1,212","3,058","3,036",527,"7,064"
15,Azuay,Total Azuay,Urbana,"131,740","127,343","2,976",227,207,657,26,304


## 2. Seleccionar la provincia de El Oro

En el tabulado original: columna `1` = provincia; `2` = cantón; `3` = total o área; `4` = viviendas ocupadas; `5` = viviendas conectadas a red pública.

In [3]:
df_datos = df.iloc[11:].copy()
df_el_oro = df_datos.loc[df_datos[1] == "El Oro"].copy()

print("Registros de El Oro (incluidos totales):", len(df_el_oro))
display(df_el_oro[[2, 3, 4, 5]].head(9))

Registros de El Oro (incluidos totales): 45


,2,3,4,5
188,Total El Oro,Total El Oro,"216,642","170,804"
189,Total El Oro,Urbana,"170,478","151,238"
190,Total El Oro,Rural,"46,164","19,566"
191,Machala,Total Machala,"92,040","80,275"
192,Machala,Urbana,"86,605","78,807"
193,Machala,Rural,"5,435","1,468"
194,Arenillas,Total Arenillas,"9,949","6,986"
195,Arenillas,Urbana,"6,592","5,712"
196,Arenillas,Rural,"3,357","1,274"


## 3. Indicador de cobertura por cantón

**Definición:**

$$
	ext{Cobertura de alcantarillado (\%)} = \frac{\text{Viviendas conectadas a red pública}}{\text{Total de viviendas particulares ocupadas}}\times 100
$$

Se selecciona solo la fila total de cada cantón para evitar contar también sus filas urbana y rural.

In [4]:
df_cantones = df_el_oro.loc[
    df_el_oro[3].str.startswith("Total") &
    (df_el_oro[2] != "Total El Oro")
].copy()

for col in [4, 5]:
    df_cantones[col] = pd.to_numeric(
        df_cantones[col].str.replace(",", "", regex=False),
        errors="raise"
    )

assert len(df_cantones) == 14, "Revisar el filtro: esperábamos 14 cantones."
assert (df_cantones[4] > 0).all(), "Se encontraron denominadores no positivos."
assert (df_cantones[5] <= df_cantones[4]).all(), "Los conectados exceden el total."

df_cantones = df_cantones.rename(columns={
    2: "canton",
    4: "total_viviendas",
    5: "viviendas_alcantarillado"
})

df_cantones["cobertura_pct"] = (
    df_cantones["viviendas_alcantarillado"] /
    df_cantones["total_viviendas"] * 100
)

display(df_cantones[["canton", "total_viviendas", "viviendas_alcantarillado", "cobertura_pct"]]
        .sort_values("cobertura_pct")
        .style.format({"cobertura_pct": "{:.2f}%"}))

,canton,total_viviendas,viviendas_alcantarillado,cobertura_pct
203,Chilla,772,366,47.41%
200,Balsas,2394,1292,53.97%
197,Atahualpa,1958,1106,56.49%
230,Las Lajas,1814,1052,57.99%
227,Zaruma,7759,4792,61.76%
218,Piñas,9378,6142,65.49%
206,El Guabo,17090,11459,67.05%
194,Arenillas,9949,6986,70.22%
209,Huaquillas,16936,12704,75.01%
224,Santa Rosa,24310,18806,77.36%


### Interpretación preliminar

**Por completar después de ejecutar:** identifica el cantón con menor y mayor cobertura y describe la diferencia en puntos porcentuales. Evita interpretar un indicador de red pública como medida total de saneamiento.

## 4. Cobertura por área urbana y rural

Para contrastar H2 seleccionamos las dos áreas de cada cantón, excluyendo los agregados provinciales. Se espera obtener **28 filas = 14 cantones × 2 áreas**.

In [5]:
df_areas = df_el_oro.loc[
    df_el_oro[3].isin(["Urbana", "Rural"]) &
    (df_el_oro[2] != "Total El Oro")
].copy()

assert len(df_areas) == 28, "Revisar los registros urbano-rurales."
assert df_areas[2].nunique() == 14, "Revisar los cantones presentes."

for col in [4, 5]:
    df_areas[col] = pd.to_numeric(
        df_areas[col].str.replace(",", "", regex=False),
        errors="raise"
    )

assert (df_areas[4] > 0).all(), "Hay denominadores no positivos."

df_areas = df_areas.rename(columns={
    2: "canton", 3: "area", 4: "total_viviendas", 5: "viviendas_alcantarillado"
})
df_areas["cobertura_pct"] = (
    df_areas["viviendas_alcantarillado"] / df_areas["total_viviendas"] * 100
)

display(df_areas[["canton", "area", "cobertura_pct"]].head(10)
        .style.format({"cobertura_pct": "{:.2f}%"}))

,canton,area,cobertura_pct
192,Machala,Urbana,91.00%
193,Machala,Rural,27.01%
195,Arenillas,Urbana,86.65%
196,Arenillas,Rural,37.95%
198,Atahualpa,Urbana,96.45%
199,Atahualpa,Rural,40.32%
201,Balsas,Urbana,67.76%
202,Balsas,Rural,34.57%
204,Chilla,Urbana,87.07%
205,Chilla,Rural,19.78%


### Comprobación: Machala

Observa si la proporción de viviendas conectadas es mayor en el área urbana o rural. Los porcentajes pueden ser muy distintos aunque pertenezcan al mismo cantón.

In [6]:
display(df_areas.loc[
    df_areas["canton"] == "Machala", ["canton", "area", "cobertura_pct"]
].style.format({"cobertura_pct": "{:.2f}%"}))

,canton,area,cobertura_pct
192,Machala,Urbana,91.00%
193,Machala,Rural,27.01%


## 5. Siguiente paso: brechas urbano-rurales en los 14 cantones

Transformaremos los datos de formato **largo** (dos filas por cantón) a **ancho** (una fila por cantón y una columna por área).

La brecha, en **puntos porcentuales (pp)**, se define como:

$$\text{Brecha}_{c} = \text{Cobertura urbana}_{c} - \text{Cobertura rural}_{c}$$

- Brecha positiva: cobertura urbana mayor.
- Brecha negativa: cobertura rural mayor.
- Brecha cercana a cero: porcentajes similares.

Se conservan valores sin redondear para el cálculo; se redondean únicamente al mostrar la tabla.

In [7]:
df_brechas = df_areas.pivot(
    index="canton",
    columns="area",
    values="cobertura_pct"
)

df_brechas["brecha_pp"] = df_brechas["Urbana"] - df_brechas["Rural"]
df_brechas = df_brechas.sort_values("brecha_pp", ascending=False)

display(df_brechas.style.format({
    "Urbana": "{:.2f}%", "Rural": "{:.2f}%", "brecha_pp": "{:+.2f} pp"
}))

area,Rural,Urbana,brecha_pp
canton,,,
Portovelo,24.52%,97.99%,+73.48 pp
Huaquillas,7.14%,75.46%,+68.32 pp
Chilla,19.78%,87.07%,+67.29 pp
Machala,27.01%,91.00%,+63.99 pp
Piñas,26.94%,88.04%,+61.10 pp
Zaruma,36.88%,96.07%,+59.19 pp
Atahualpa,40.32%,96.45%,+56.14 pp
Marcabelí,41.86%,92.72%,+50.86 pp
Arenillas,37.95%,86.65%,+48.70 pp


### Preguntas de interpretación

1. ¿En cuántos de los 14 cantones la brecha es positiva?
En Portovelo, la zona urbana tiene mayor cobertura
2. ¿Cuál presenta la mayor brecha? ¿Y la menor?
La mayor brecha se presenta en Portovelo también, y la menor la tiene Pasaje.
3. ¿Hay un cantón con cobertura rural mayor que la urbana?
Ninguno
4. ¿El patrón respalda H2 en El Oro? **Describir los hallazgos sin atribuir causas**.

El análisis de los datos del Censo de Población y Vivienda 2022 muestra que los 14 cantones de la provincia de El Oro presentan una mayor cobertura de conexión a la red pública de alcantarillado en sus áreas urbanas que en sus áreas rurales.

Portovelo registra la mayor brecha urbano-rural (73,48 puntos porcentuales), seguido de Huaquillas (68,32 pp) y Chilla (67,29 pp). Por su parte, Pasaje presenta la menor brecha (28,46 pp).

Estos resultados son consistentes con la hipótesis H2 para el indicador estudiado. Sin embargo, las brechas reflejan diferencias de cobertura entre áreas y no permiten establecer sus causas ni medir por sí solas las condiciones generales de saneamiento de cada cantón.

El análisis deberá complementarse con indicadores de otros servicios básicos y con una revisión de las características territoriales.


**Por hacer:** calcular la cobertura urbana y rural agregada de El Oro a partir de sus conteos (no promediar sin ponderar los porcentajes cantonales), y después integrar agua, electricidad y basura.

In [8]:
df_provincia = df_el_oro[
    df_el_oro[2] == "Total El Oro"
].copy()

df_provincia[[2, 3, 4, 5]]

,2,3,4,5
188,Total El Oro,Total El Oro,"216,642","170,804"
189,Total El Oro,Urbana,"170,478","151,238"
190,Total El Oro,Rural,"46,164","19,566"


In [9]:
for col in [4, 5]:
    df_provincia[col] = pd.to_numeric(
        df_provincia[col].str.replace(",", "", regex=False),
        errors="raise"
    )

In [10]:
df_provincia["cobertura_pct"] = (
    df_provincia[5] / df_provincia[4]
) * 100

In [11]:
df_provincia[[3, 4, 5, "cobertura_pct"]]

,3,4,5,cobertura_pct
188,Total El Oro,216642,170804,78.841591
189,Urbana,170478,151238,88.714086
190,Rural,46164,19566,42.383676


In [12]:
df_areas = df_areas.rename(columns={
    2: "Cantón",
    3: "Área",
    4: "Viviendas ocupadas",
    5: "Viviendas con alcantarillado",
    "cobertura_pct": "Cobertura %"
})

In [13]:
df_areas.columns.tolist()

[0,
 1,
 'canton',
 'area',
 'total_viviendas',
 'viviendas_alcantarillado',
 6,
 7,
 8,
 9,
 10,
 11,
 'Cobertura %']

In [14]:
cobertura_urbana = df_provincia.loc[
    df_provincia[3] == "Urbana",
    "cobertura_pct"
].iloc[0]

In [15]:
cobertura_rural = df_provincia.loc[
    df_provincia[3] == "Rural",
    "cobertura_pct"
].iloc[0]

In [16]:
brecha_provincial = cobertura_urbana - cobertura_rural

print(f"Brecha provincial: {brecha_provincial:.2f} pp")

Brecha provincial: 46.33 pp


¿Qué significa este resultado?
Que, considerando todas las viviendas particulares ocupadas de la provincia, la conexión a la red pública de alcantarillado es considerablemente mayor en las zonas urbanas.

In [17]:
promedio_brechas = df_brechas["brecha_pp"].mean()

print(f"Promedio de brechas cantonales: {promedio_brechas:.2f} pp")
print(f"Brecha provincial: {brecha_provincial:.2f} pp")

Promedio de brechas cantonales: 52.20 pp
Brecha provincial: 46.33 pp


### PROMEDIO DE BRECHAS CANTONALES

52,20 pp
Cada cantón tiene el mismo peso.

BRECHA PROVINCIAL
46,33 pp
Se utilizan los totales de viviendas por área.

A partir del Censo de Población y Vivienda 2022, se obtuvo una brecha urbano-rural de 46,33 puntos porcentuales en la cobertura de conexión a la red pública de alcantarillado para el conjunto de viviendas particulares ocupadas de la provincia de El Oro.

Por otra parte, el promedio aritmético de las brechas urbano-rurales de sus 14 cantones fue de 52,20 puntos porcentuales.

La diferencia entre ambos resultados se debe a que el promedio cantonal otorga el mismo peso a cada cantón, mientras que la brecha provincial se calcula a partir de los totales de viviendas conectadas y viviendas ocupadas en cada área.

Estos resultados muestran la importancia de distinguir entre indicadores agregados provinciales y promedios de indicadores cantonales al estudiar desigualdades territoriales.

In [18]:
print(df_areas.columns.tolist())

[0, 1, 'canton', 'area', 'total_viviendas', 'viviendas_alcantarillado', 6, 7, 8, 9, 10, 11, 'Cobertura %']


In [19]:
df_validacion = df_areas.groupby("canton")["total_viviendas"].sum()

df_validacion

canton
Arenillas      9949
Atahualpa      1958
Balsas         2394
Chilla          772
El Guabo      17090
Huaquillas    16936
Las Lajas      1814
Machala       92040
Marcabelí      2095
Pasaje        25860
Piñas          9378
Portovelo      4287
Santa Rosa    24310
Zaruma         7759
Name: total_viviendas, dtype: int64

In [21]:
print(df_cantones.columns.tolist())

[0, 1, 'canton', 3, 'total_viviendas', 'viviendas_alcantarillado', 6, 7, 8, 9, 10, 11, 'cobertura_pct']


In [22]:
df_comprobacion = df_cantones[
    ["canton", "total_viviendas"]
].copy()

df_comprobacion["suma_areas"] = (
    df_comprobacion["canton"].map(df_validacion)
)

df_comprobacion["diferencia"] = (
    df_comprobacion["total_viviendas"]
    - df_comprobacion["suma_areas"]
)

df_comprobacion

,canton,total_viviendas,suma_areas,diferencia
191,Machala,92040,92040,0
194,Arenillas,9949,9949,0
197,Atahualpa,1958,1958,0
200,Balsas,2394,2394,0
203,Chilla,772,772,0
206,El Guabo,17090,17090,0
209,Huaquillas,16936,16936,0
212,Marcabelí,2095,2095,0
215,Pasaje,25860,25860,0
218,Piñas,9378,9378,0
